# 01 - X5 RetailHero Data Audit

Exploration and reporting notebook for the raw X5 RetailHero dataset.

All audit logic lives in `src/promolift/validation/data_audit.py` and
`src/promolift/data/loader.py` -- this notebook only calls into it and
displays results. It does not duplicate any business logic.

This covers the *structural* audit only: inventory, schemas, row/column
counts, null rates, numeric/date ranges, and duplicate-key checks. Domain
checks (client/product foreign-key integrity, transaction date integrity,
treatment/control balance, outcome distribution, ATE sanity check) are
deferred to a later, more targeted audit.

In [ ]:
import polars as pl

from promolift.data.loader import Dataset
from promolift.validation.data_audit import audit_dataset, column_cardinality, inventory

## Dataset inventory

Confirms every registered raw file exists and reports its size. This only
calls `Path.stat()` -- no file is scanned or loaded here.

In [ ]:
inventory_df = pl.DataFrame(
    [
        {
            "name": entry.name,
            "exists": entry.exists,
            "size_mb": round(entry.size_bytes / 1_000_000, 2) if entry.size_bytes else None,
            "path": str(entry.path),
        }
        for entry in inventory()
    ]
)
inventory_df

## Structural audit: small files

`clients`, `products`, and the `uplift_*` files are small enough to audit
eagerly here. `purchases.csv` (~4.2 GB) is audited separately below, since
it takes noticeably longer to scan.

In [ ]:
small_datasets = [
    Dataset.CLIENTS,
    Dataset.PRODUCTS,
    Dataset.UPLIFT_TRAIN,
    Dataset.UPLIFT_TEST,
    Dataset.UPLIFT_SAMPLE_SUBMISSION,
]

# client_id is the expected natural key for these files; used only to check
# for duplicates, not assumed as ground truth about the schema.
key_columns_by_dataset = {
    Dataset.CLIENTS: ["client_id"],
    Dataset.UPLIFT_TRAIN: ["client_id"],
    Dataset.UPLIFT_TEST: ["client_id"],
    Dataset.UPLIFT_SAMPLE_SUBMISSION: ["client_id"],
}

audits = {
    dataset: audit_dataset(dataset, key_columns=key_columns_by_dataset.get(dataset))
    for dataset in small_datasets
}

pl.DataFrame(
    [
        {
            "name": audit.name,
            "row_count": audit.row_count,
            "column_count": audit.column_count,
            "duplicate_key_count": audit.duplicate_key_count,
        }
        for audit in audits.values()
    ]
)

### Column-level detail

Null counts/rates and numeric or date ranges per column, for each small
dataset.

In [ ]:
for dataset, audit in audits.items():
    print(f"--- {dataset.value} ---")
    display(
        pl.DataFrame(
            [
                {
                    "column": col.name,
                    "dtype": col.dtype,
                    "null_count": col.null_count,
                    "null_rate": round(col.null_rate, 4),
                    "min": col.min_value,
                    "max": col.max_value,
                }
                for col in audit.columns
            ]
        )
    )

## Structural audit: `purchases.csv` (large file)

This scans the full ~4.2 GB file lazily via Polars (`scan_csv`) to compute
row counts, null rates, and numeric/date ranges in a single pass. No
duplicate-key check is run by default here, since a `group_by` over the
full file is comparatively expensive -- pass `key_columns` explicitly if
needed. Expect this cell to take noticeably longer than the ones above.

In [ ]:
purchases_audit = audit_dataset(Dataset.PURCHASES)

print("row_count:", purchases_audit.row_count)
print("column_count:", purchases_audit.column_count)

pl.DataFrame(
    [
        {
            "column": col.name,
            "dtype": col.dtype,
            "null_count": col.null_count,
            "null_rate": round(col.null_rate, 4),
            "min": col.min_value,
            "max": col.max_value,
        }
        for col in purchases_audit.columns
    ]
)

## Key cardinality checks

Example: how many distinct clients and products appear in `purchases.csv`,
compared against the `clients` and `products` reference files.

In [ ]:
purchases_cardinality = column_cardinality(
    Dataset.PURCHASES, ["client_id", "product_id", "store_id"]
)
purchases_cardinality

## Deferred to a later, targeted audit

- Client/product foreign-key integrity between `purchases`, `clients`, and `products`
- Transaction date integrity (ordering, gaps, out-of-range dates)
- Treatment/control balance in `uplift_train`
- Outcome (`target`) distribution by treatment group
- Raw experimental treatment effect / ATE sanity check